# Feature IC checks - 1h candles

This notebook tests candidate alpha features from `bot/features.py` against a configurable forward return horizon. It is research-only; live trading should call the same feature functions on fresh in-memory Binance candles.

Default run: `HORIZON = 6` one-hour bars, alpha windows `(6, 12, 24)`, microstructure interaction window `50` hours.

Challenge / caveat: pooled IC can look good while being unusable for a rank-based long-only bot. For asset selection, the most relevant table here is **cross-sectional IC by timestamp**, because the bot chooses among assets at the same time.

## Feature definitions

- `vol_adj_momentum_W`: `log_return_W / realized_vol_W`.
- `residual_momentum_W`: coin momentum minus the same-timestamp universe mean momentum.
- `relative_strength_W`: same-timestamp percentile rank of momentum.
- `trend_persistence_W`: fraction of positive one-bar returns over `W`, centered to `[-1, 1]`.
- `breakout_quality_W`: close above prior rolling high, scaled by volume confirmation.
- `momentum_low_vpin_W` and `momentum_low_roll_impact_W`: volatility-adjusted momentum scaled by low microstructure-risk rank.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = next(parent for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (parent / "bot").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bot.features import add_alpha_features, alpha_feature_columns  # noqa: E402
from bot.forward_ic import add_forward_return, compute_univariate_ics  # noqa: E402
from bot.microstructure import compute_microstructure_measures  # noqa: E402

INTERVAL = "1h"
HORIZON = 6
ALPHA_WINDOWS = (6, 12, 24)
MICROSTRUCTURE_WINDOW = 50
MIN_PERIODS = 30
PREFIX = f"feature_checks_1h_h{HORIZON}"
RAW_PATH = NOTEBOOK_DIR.parent / "microstructure" / "ic_checks_1h_candles_raw.csv"

FEATURES_PATH = NOTEBOOK_DIR / f"{PREFIX}_features.csv"
POOLED_PATH = NOTEBOOK_DIR / f"{PREFIX}_pooled.csv"
PER_PAIR_PATH = NOTEBOOK_DIR / f"{PREFIX}_per_pair.csv"
CS_BY_TIME_PATH = NOTEBOOK_DIR / f"{PREFIX}_cross_sectional_by_time.csv"
CS_SUMMARY_PATH = NOTEBOOK_DIR / f"{PREFIX}_cross_sectional_summary.csv"
METADATA_PATH = NOTEBOOK_DIR / f"{PREFIX}_metadata.json"

In [2]:
def build_feature_frame(raw: pd.DataFrame):
    parts = []
    for _, group in raw.groupby("pair", sort=False):
        group = group.sort_values("open_time").reset_index(drop=True)
        group = compute_microstructure_measures(group, window=MICROSTRUCTURE_WINDOW)
        parts.append(group)
    frame = pd.concat(parts, ignore_index=True)
    frame = add_alpha_features(frame, windows=ALPHA_WINDOWS)
    frame = add_forward_return(
        frame,
        horizon=HORIZON,
        price_col="close",
        target_col=f"forward_return_{HORIZON}",
        group_col="pair",
        sort_col="open_time",
    )
    return frame


def compute_feature_ic_tables(features: pd.DataFrame, feature_cols: list[str]):
    target_col = f"forward_return_{HORIZON}"
    pooled = compute_univariate_ics(
        features,
        feature_cols=feature_cols,
        target_col=target_col,
        methods=("pearson", "spearman"),
        horizon=HORIZON,
        min_periods=MIN_PERIODS,
    )
    per_pair = compute_univariate_ics(
        features,
        feature_cols=feature_cols,
        target_col=target_col,
        methods=("pearson", "spearman"),
        horizon=HORIZON,
        min_periods=MIN_PERIODS,
        group_col="pair",
    )
    cross_sectional_by_time = compute_univariate_ics(
        features,
        feature_cols=feature_cols,
        target_col=target_col,
        methods=("pearson", "spearman"),
        horizon=HORIZON,
        min_periods=10,
        group_col="open_time",
    )
    cross_sectional_summary = (
        cross_sectional_by_time.groupby(["feature", "method"])["ic"]
        .agg(["mean", "median", "std", "count"])
        .reset_index()
    )
    positive_rate = (
        cross_sectional_by_time.assign(is_positive=cross_sectional_by_time["ic"] > 0)
        .groupby(["feature", "method"])["is_positive"]
        .mean()
        .reset_index(name="positive_rate")
    )
    cross_sectional_summary = cross_sectional_summary.merge(
        positive_rate,
        on=["feature", "method"],
        how="left",
    )
    return pooled, per_pair, cross_sectional_by_time, cross_sectional_summary


raw = pd.read_csv(RAW_PATH)
features = build_feature_frame(raw)
feature_cols = [col for col in alpha_feature_columns(ALPHA_WINDOWS) if col in features.columns]
pooled, per_pair, cross_sectional_by_time, cross_sectional_summary = compute_feature_ic_tables(
    features,
    feature_cols,
)
metadata = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "raw_source": str(RAW_PATH),
    "interval": INTERVAL,
    "horizon": HORIZON,
    "alpha_windows": list(ALPHA_WINDOWS),
    "microstructure_window": MICROSTRUCTURE_WINDOW,
    "pairs": int(raw["pair"].nunique()),
    "raw_rows": int(len(raw)),
    "feature_rows": int(len(features)),
    "feature_count": len(feature_cols),
}

features.to_csv(FEATURES_PATH, index=False)
pooled.to_csv(POOLED_PATH, index=False)
per_pair.to_csv(PER_PAIR_PATH, index=False)
cross_sectional_by_time.to_csv(CS_BY_TIME_PATH, index=False)
cross_sectional_summary.to_csv(CS_SUMMARY_PATH, index=False)
METADATA_PATH.write_text(json.dumps(metadata, indent=2, sort_keys=True))
metadata

{'generated_at': '2026-06-05T21:26:03.604448+00:00',
 'raw_source': '/home/weiyuanoh/projects/roostoo-bot-v2/notebooks/microstructure/ic_checks_1h_candles_raw.csv',
 'interval': '1h',
 'horizon': 6,
 'alpha_windows': [6, 12, 24],
 'microstructure_window': 50,
 'pairs': 43,
 'raw_rows': 43000,
 'feature_rows': 43000,
 'feature_count': 27}

In [3]:
def svg_bar_chart(df, value_col, label_col, title, width=1040, row_h=25, left=330, right=90, limit=20):
    rows = df[[label_col, value_col]].dropna().copy().sort_values(value_col)
    if limit and len(rows) > limit:
        low = rows.head(limit // 2)
        high = rows.tail(limit - len(low))
        rows = pd.concat([low, high]).sort_values(value_col)
    height = 50 + row_h * len(rows)
    plot_w = width - left - right
    max_abs = max(float(rows[value_col].abs().max()), 0.001)
    zero_x = left + plot_w / 2
    scale = (plot_w / 2) / max_abs
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
    parts.append('<rect width="100%" height="100%" fill="#fff"/>')
    parts.append(f'<text x="12" y="24" font-family="Arial" font-size="16" font-weight="700">{title}</text>')
    parts.append(f'<line x1="{zero_x:.1f}" y1="38" x2="{zero_x:.1f}" y2="{height-12}" stroke="#555"/>')
    for i, row in enumerate(rows.itertuples(index=False)):
        label = str(getattr(row, label_col)).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        value = float(getattr(row, value_col))
        y = 46 + i * row_h
        bar_w = abs(value) * scale
        x = zero_x if value >= 0 else zero_x - bar_w
        color = "#2f7d32" if value >= 0 else "#b23a3a"
        parts.append(f'<text x="12" y="{y+13}" font-family="Arial" font-size="11">{label}</text>')
        parts.append(f'<rect x="{x:.1f}" y="{y}" width="{bar_w:.1f}" height="16" fill="{color}" opacity="0.82"/>')
        tx = zero_x + bar_w + 6 if value >= 0 else zero_x - bar_w - 58
        parts.append(f'<text x="{tx:.1f}" y="{y+13}" font-family="Arial" font-size="11">{value:+.4f}</text>')
    parts.append("</svg>")
    return "".join(parts)

## Pooled IC

Pooled IC stacks all pair-time observations. Use it as a broad diagnostic, not a final ranking metric.

In [4]:
pooled.assign(abs_ic=pooled["ic"].abs()).sort_values("abs_ic", ascending=False).head(20)

,feature,target,method,ic,n,horizon,group,rank_method,abs_ic
21,realized_vol_12,forward_return_6,spearman,-0.099316,42226,6,None,average,0.099316
39,realized_vol_24,forward_return_6,spearman,-0.096453,41710,6,None,average,0.096453
3,realized_vol_6,forward_return_6,spearman,-0.095052,42484,6,None,average,0.095052
0,log_return_6,forward_return_6,pearson,0.061880,42484,6,None,NaN,0.061880
2,realized_vol_6,forward_return_6,pearson,-0.061306,42484,6,None,NaN,0.061306
48,residual_momentum_24,forward_return_6,pearson,0.050975,41710,6,None,NaN,0.050975
41,vol_adj_momentum_24,forward_return_6,spearman,-0.048038,41710,6,None,average,0.048038
38,realized_vol_24,forward_return_6,pearson,-0.047061,41710,6,None,NaN,0.047061
20,realized_vol_12,forward_return_6,pearson,-0.045465,42226,6,None,NaN,0.045465
46,relative_strength_24,forward_return_6,pearson,0.044185,41710,6,None,NaN,0.044185


In [5]:
pooled_plot = pooled[pooled["method"].eq("spearman")].copy()
pooled_plot["label"] = pooled_plot["feature"]
display(HTML(svg_bar_chart(pooled_plot, "ic", "label", f"Pooled Spearman IC vs forward {HORIZON}h return")))

## Cross-sectional IC by timestamp

This is the main alpha diagnostic for a ranker. Each timestamp computes IC across the asset universe, then this table summarizes those timestamp-level ICs.

In [6]:
cross_sectional_summary[cross_sectional_summary["method"].eq("spearman")].sort_values("mean", ascending=False)

,feature,method,mean,median,std,count,positive_rate
51,vol_adj_momentum_24,spearman,0.048972,0.046022,0.215545,970,0.582
21,momentum_low_vpin_24,spearman,0.046260,0.047722,0.184854,895,0.532
15,momentum_low_roll_impact_24,spearman,0.030168,0.035035,0.207398,943,0.534
19,momentum_low_vpin_12,spearman,0.019445,0.016309,0.186345,895,0.480
45,trend_persistence_24,spearman,0.018267,0.018686,0.184592,971,0.521
9,log_return_24,spearman,0.018222,0.029825,0.271248,970,0.541
33,relative_strength_24,spearman,0.018222,0.029825,0.271248,970,0.541
39,residual_momentum_24,spearman,0.018222,0.029825,0.271248,970,0.541
49,vol_adj_momentum_12,spearman,0.012721,0.015101,0.204740,982,0.520
23,momentum_low_vpin_6,spearman,0.009157,0.008005,0.183300,895,0.463


In [7]:
cs_plot = cross_sectional_summary[cross_sectional_summary["method"].eq("spearman")].copy()
cs_plot["label"] = cs_plot["feature"]
display(HTML(svg_bar_chart(cs_plot, "mean", "label", f"Mean cross-sectional Spearman IC, h={HORIZON}")))

## Per-pair IC summary

Per-pair IC checks whether a feature has broadly similar behavior across symbols or is driven by a few names.

In [8]:
per_pair_summary = (
    per_pair.groupby(["feature", "method"])["ic"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
    .sort_values(["method", "mean"], ascending=[True, False])
)
per_pair_summary


,feature,method,mean,median,std,count
16,momentum_low_roll_impact_6,pearson,0.035451,0.027105,0.062060,43
22,momentum_low_vpin_6,pearson,0.033129,0.028477,0.061567,43
10,log_return_6,pearson,0.033061,0.038308,0.074397,43
32,relative_strength_24,pearson,0.032162,0.031675,0.093845,43
52,vol_adj_momentum_6,pearson,0.031767,0.022823,0.058572,43
46,trend_persistence_6,pearson,0.028057,0.026964,0.066128,43
30,relative_strength_12,pearson,0.016205,0.001258,0.086086,43
12,momentum_low_roll_impact_12,pearson,0.015492,0.018493,0.076270,43
6,log_return_12,pearson,0.013239,-0.000791,0.085847,43
18,momentum_low_vpin_12,pearson,0.013177,0.004537,0.080309,43


## First-pass read

Read this notebook in order: first cross-sectional IC, then pooled IC, then per-pair stability. A feature is more credible if it has the same sign across cross-sectional mean/median, pooled Spearman, and per-pair median. Single large per-pair ICs are hypothesis generators, not conclusions.